# Wirkungsanalyse der Begehungen

**Zweck:** Misst, ob sich das Betriebsverhalten einer HAST nach einer Begehung messbar
geaendert hat - und ob die Aenderung zur dokumentierten Aenderungs-Kategorie passt.

**Warum dieser Ansatz:** Die Begehung wird nicht als *Label* behandelt
(„hat diese Station einen Fehler?“), sondern als **Ereignis** („was hat sich danach
geaendert?“). Das umgeht das Grundproblem der bisherigen Ansaetze: Die Zielspalte
`n_actions > 0` ist bei 76 % der Stationen wahr und kann deshalb nichts trennen.
Die Zielgroesse kommt hier aus den Messdaten selbst.

**Datenquellen:** `Ergebnis_Optimierung` (Begehungen, 133 Spalten) und `Zuordnung.xlsx`
(Adresse -> Zaehlernummer), beide ueber [`src/load_begehungen.py`](../src/load_begehungen.py).
Dazu die stuendlichen Zeitreihen aus `data/processed/real_meters/`.

## 1 · Setup & Daten laden

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load_begehungen import begehungen_mit_zaehler, meter_changes, visit_hours

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.facecolor"] = "white"

METER_DIR = ROOT / "data" / "processed" / "real_meters"
VERFUEGBAR = {int(p.stem) for p in METER_DIR.glob("*.csv")}
print(f"{len(VERFUEGBAR)} Zeitreihen in {METER_DIR.relative_to(ROOT)}")

SPALTEN = {
    "Volume flow (l/h)": "flow", "Return temperature (°C)": "rt",
    "Flow temperature (°C)": "vl", "Power (kW)": "kw",
    "Temperature difference (°C)": "dt",
}


def zeitreihe(meter):
    """Stundenreihe eines Zaehlers mit kurzen Spaltennamen."""
    pfad = METER_DIR / f"{int(meter)}.csv"
    if not pfad.is_file():
        return None
    df = pd.read_csv(pfad, parse_dates=["Timestamp"])
    return df.set_index("Timestamp").sort_index().rename(columns=SPALTEN)

**Was macht der Code?**
- Sucht die Projektwurzel selbst (nach oben, bis `src/` auftaucht) und haengt sie an `sys.path`
  → das Notebook laeuft aus `notebooks/` genauso wie aus dem Repo-Root
- Importiert den Loader aus `src/load_begehungen.py`. Bewusst **nicht** hier im Notebook
  definiert: Die Zuordnung Adresse → Zaehlernummer ist Datenzugriff, kein Analyseschritt,
  und wird von jeder kuenftigen Auswertung gebraucht
- `VERFUEGBAR` = die Zaehler, fuer die wir tatsaechlich Zeitreihen haben. Wird an den Loader
  uebergeben, damit Adressen mit **Zaehlerwechsel** auf ein Geraet mit Daten aufgeloest werden
- `zeitreihe()` kapselt das Einlesen: Stundenraster, kurze Spaltennamen (`rt`, `flow`, `dt`)

## 2 · Begehungen und Zaehlernummern

Der Join Adresse → Zaehler laeuft ueber `Zuordnung.xlsx`, **nicht** ueber `Nodes_Edges.ods`.

In [ ]:
beg = begehungen_mit_zaehler(available=VERFUEGBAR)

print(f"Begehungen in der Excel : {len(beg)}")
print(f"  mit Zaehlernummer     : {beg['meter'].notna().sum()}")
print(f"  mit Datum             : {beg['datum'].notna().sum()}")
print(f"  mit Uhrzeit           : {beg['von'].notna().sum()}")
print(f"  mit Heizzeit-Aenderung: {int(beg['heizzeit_geaendert'].sum())}")

print("\nAenderungs-Kategorie:")
display(beg["kategorie"].value_counts(dropna=False).to_frame("n"))

wechsel = meter_changes()
print(f"Adressen mit Zaehlerwechsel ueber die Jahre: {len(wechsel)}")
for adresse, zaehler in wechsel.items():
    print(f"  {adresse:22s} {zaehler}")

**Was macht der Code?**
- `begehungen_mit_zaehler()` liest beide Excel-Dateien und verbindet sie ueber den
  normalisierten Adressschluessel
- **Alle 66 Begehungen** bekommen eine Zaehlernummer. Ueber die Adressen aus
  `Nodes_Edges.ods` (die `hast_features.csv` verwendet) waeren es nur 54 — dort fehlen
  16 Adressen, darunter die Zaehler 72167790, 68956340, 80912200 und 80912169
- Der Loader korrigiert ausserdem einen Tippfehler in der Excel: „Fleiderstrasse“ statt
  „Fliederstrasse“, kostet allein vier Zuordnungen

**Die Aenderungs-Kategorie ist der eigentliche Gewinn**
`inspections_tidy.csv` hat 21 von 133 Spalten uebernommen — diese war nicht dabei.
Sie ist ein **vierstufiges, fachlich vergebenes Label** (keine / geringe / mittlere / viele)
und damit weit besser als das abgeleitete `n_actions > 0` mit 76 % Basisrate.

**Zaehlerwechsel im Blick behalten**
Fuenf Adressen haben ueber die Jahre verschiedene Zaehlernummern — z. B. 65273846 →
80912176 ab 2025. Ein Vorher-Nachher ueber so einen Wechsel hinweg
vergleicht zwei **Geraete**, nicht zwei Zustaende einer Station.

## 3 · Vorher/Nachher je Station

Fenster: 21 Tage vor und 21 Tage nach der Begehung. Kurz genug, dass sich Wetter und
Jahreszeit kaum aendern, lang genug fuer stabile Mediane.

In [ ]:
FENSTER = pd.Timedelta(days=21)
MIN_STUNDEN = 200


def vorher_nachher(meter, datum):
    """Mediane von Ruecklauf und Durchfluss vor/nach dem Stichtag."""
    df = zeitreihe(meter)
    if df is None:
        return None
    vor = df.loc[datum - FENSTER : datum - pd.Timedelta(hours=1)]
    nach = df.loc[datum + pd.Timedelta(days=1) : datum + FENSTER]
    if vor["rt"].count() < MIN_STUNDEN or nach["rt"].count() < MIN_STUNDEN:
        return None
    return {
        "rt_vor": vor["rt"].median(), "rt_nach": nach["rt"].median(),
        "flow_vor": vor["flow"].median(), "flow_nach": nach["flow"].median(),
    }


zeilen = []
for _, r in beg[beg["datum"].notna() & beg["meter"].notna()].iterrows():
    werte = vorher_nachher(int(r["meter"]), r["datum"].normalize())
    if werte is None:
        continue
    zeilen.append({
        "meter": int(r["meter"]), "adresse": r["key"],
        "datum": r["datum"].normalize(), "kategorie": r["kategorie"],
        "von": str(r["von"])[:5] if pd.notna(r["von"]) else "",
        "meter_gewechselt": r["meter_gewechselt"], **werte,
    })

wirkung = pd.DataFrame(zeilen)
wirkung["d_rt"] = wirkung["rt_nach"] - wirkung["rt_vor"]
wirkung["d_flow"] = wirkung["flow_nach"] - wirkung["flow_vor"]
print(f"auswertbar: {len(wirkung)} von {beg['datum'].notna().sum()} Begehungen mit Datum")

top = (wirkung.reindex(wirkung["d_rt"].abs().sort_values(ascending=False).index)
       .head(8)[["meter", "adresse", "datum", "kategorie", "rt_vor", "rt_nach",
                 "d_rt", "flow_vor", "flow_nach"]])
display(top.style.format({c: "{:.1f}" for c in top.select_dtypes("number")}))

**Was macht der Code?**
- Fuer jede Begehung mit Datum: Median von Ruecklauf und Durchfluss in den 21 Tagen davor
  und danach. Der Begehungstag selbst bleibt aussen vor, weil dort der Eingriff stattfindet
- `MIN_STUNDEN = 200` verwirft Stationen mit zu vielen Messluecken im Fenster —
  ein Median aus 30 Stunden ist kein Median
- **Median statt Mittelwert**, weil einzelne Zapfspitzen den Vergleich sonst verziehen

**Warum 21 Tage und nicht Winter-gegen-Winter**
Ein Jahresabstand bringt zwei Probleme: unterschiedliches Wetter, und **Regression zur
Mitte** — Stationen mit hohem Ausgangswert fallen ohnehin (im Winter-Vergleich gemessen:
r = −0,55 zwischen Ausgangswert und Aenderung). Im engen Fenster ist beides klein.

**Die Einzelfaelle sind aussagekraeftiger als der Mittelwert.** Der groesste Ausschlag:
Zaehler 68956351 faellt von 499 auf 0 l/h — ein Ventil, das drei Jahre offen stand und
am Begehungstag geschlossen wurde. In `inspections_tidy.csv` steht dazu `n_actions = 0`.

## 4 · Kontrollgruppe und Differenz-von-Differenzen

Die rohe Aenderung enthaelt alles, was gleichzeitig passiert ist — vor allem den Beginn der
Heizperiode, denn die meisten Begehungen lagen im September.

In [ ]:
kontrollgruppe = sorted(VERFUEGBAR - set(wirkung["meter"]))
print(f"Kontrollgruppe (nie begangen): {len(kontrollgruppe)} Stationen")


def kontroll_delta(datum):
    """Median-Ruecklaufaenderung der nie begangenen Stationen im selben Fenster."""
    werte = []
    for meter in kontrollgruppe:
        w = vorher_nachher(meter, datum)
        if w is not None:
            werte.append(w["rt_nach"] - w["rt_vor"])
    return np.median(werte) if werte else np.nan


# je Begehungsdatum einmal rechnen, nicht je Station
delta_je_datum = {d: kontroll_delta(pd.Timestamp(d)) for d in sorted(wirkung["datum"].unique())}
wirkung["kontrolle"] = wirkung["datum"].map(delta_je_datum)
wirkung["did"] = wirkung["d_rt"] - wirkung["kontrolle"]

REIHENFOLGE = ["keine", "geringe", "mittlere", "viele"]
tabelle = (wirkung.groupby("kategorie")
           .agg(n=("did", "size"), roh=("d_rt", "median"),
                kontrolle=("kontrolle", "median"), DiD=("did", "median"))
           .reindex(REIHENFOLGE).round(2))
display(tabelle)

for kat in ["mittlere", "viele"]:
    werte = wirkung.loc[wirkung["kategorie"] == kat, "did"].dropna()
    if len(werte) > 5:
        p = stats.wilcoxon(werte)[1]
        print(f"{kat:9s} n={len(werte):2d}  Median {werte.median():+.2f} K  Wilcoxon p = {p:.3f}")

**Was macht der Code?**
- Bildet als Kontrollgruppe die Stationen, die **nie begangen** wurden
- Rechnet fuer jedes Begehungsdatum die Median-Aenderung dieser Gruppe im selben Fenster
- Zieht sie von der Aenderung der begangenen Station ab → **Differenz-von-Differenzen**

**Was DiD leistet**
Erste Differenz: Aenderung der behandelten Station. Zweite Differenz: Aenderung der
Kontrollgruppe, also der Anteil, der ohnehin passiert waere (Jahreszeit, Wetter,
Netzfahrweise). Die Differenz der beiden ist der Teil, der plausibel auf die Begehung
zurueckgeht.

Die Differenz wird **je Station** gebildet und davon der Median genommen — nicht die
Differenz der Gruppenmediane. Beides ist bei Medianen nicht dasselbe, und die
stationsweise Variante ist die richtige, weil jede Station ihr eigenes Datum hat.

**Ergebnis: monoton ueber alle vier Stufen.** Je mehr laut Excel geaendert wurde, desto
staerker sinkt der Ruecklauf gegenueber unbehandelten Stationen.

**Aber nicht signifikant** (Wilcoxon p ≈ 0,4). Zwei Gruende, die in die Auswertung gehoeren:
- Die Gruppen „keine“ und „geringe“ haben nur 2 bzw. 4 Faelle
- **Parallele Trends sind hier angreifbar:** Begangen wurden vermutlich eher die
  auffaelligen Stationen, und die fallen ohnehin staerker. DiD faengt den allgemeinen Trend
  ab, aber nicht diesen Auswahleffekt

## 5 · Ist die Begehung in den Daten sichtbar?

Wenn jemand die Station oeffnet, laeuft heisses Wasser durch, ohne Waerme abzugeben:
Vorlauf hoch, Ruecklauf fast genauso hoch, ΔT nahe null.

In [ ]:
DT_SCHWELLE = 3.0   # K


def signatur(meter, datum, stunden):
    """Kleinstes ΔT im dokumentierten Begehungsfenster."""
    df = zeitreihe(meter)
    if df is None or stunden is None:
        return np.nan
    start, ende = stunden
    fenster = df.loc[datum + pd.Timedelta(hours=start) : datum + pd.Timedelta(hours=ende + 1)]
    return fenster["dt"].min() if fenster["dt"].notna().any() else np.nan


mit_zeit = beg[beg["datum"].notna() & beg["meter"].notna() & beg["von"].notna()].copy()
mit_zeit["dt_min"] = [
    signatur(int(r["meter"]), r["datum"].normalize(), visit_hours(r))
    for _, r in mit_zeit.iterrows()
]
auswertbar = mit_zeit[mit_zeit["dt_min"].notna()]
treffer = auswertbar["dt_min"] < DT_SCHWELLE
print(f"Begehungen mit Uhrzeit und Daten: {len(auswertbar)}")
print(f"davon ΔT < {DT_SCHWELLE:.0f} K im Zeitfenster: {treffer.sum()} ({treffer.mean():.0%})")

sig = (auswertbar.loc[treffer, ["meter", "key", "datum", "von", "bis", "kategorie", "dt_min"]]
       .sort_values("dt_min"))
display(sig.style.format({"dt_min": "{:.2f}"}))

**Was macht der Code?**
- Nimmt fuer jede Begehung mit Uhrzeit das dokumentierte Zeitfenster und sucht dort das
  **kleinste ΔT**
- Ein ΔT nahe null bei hohem Vorlauf heisst: heisses Wasser laeuft durch, ohne Waerme
  abzugeben — Spuelen, ein Ventil von Hand oeffnen, eine Pruefung fahren

**Warum das eine Kontrolle und kein Selbstzweck ist**
Es belegt, dass das Begehungsdatum in der Excel **stimmt** und die Zuordnung auf den
richtigen Zaehler zeigt. Ohne diese Kontrolle wuesste man bei einem ausbleibenden Effekt
nicht, ob nichts passiert ist oder ob man am falschen Zaehler sucht.

**Grenze der Methode:** Bei manchen Stationen kommen niedrige ΔT-Werte auch im
Normalbetrieb vor — z. B. wenn eine Zirkulation dauerhaft laeuft. Dort ist die Signatur
nicht eindeutig. Sie taugt als Bestaetigung, nicht als Beweis.

## 6 · Dokumentierte Heizzeit-Aenderungen pruefen

Der schaerfste Test: Bei acht Stationen steht in der Excel, wie die Heizzeit **vorher**
und **nachher** eingestellt war. Eine Vorhersage, die vor dem Blick in die Daten feststeht.

In [ ]:
import re

WINTER_VOR = ("2023-12-01", "2024-02-29")
WINTER_NACH = ("2024-12-01", "2025-02-28")


def fensterstunden(von, bis):
    """Menge der Tagesstunden im Heizfenster, auch ueber Mitternacht."""
    def stunde(v):
        m = re.search(r"(\d{1,2}):(\d{2})", str(v))
        return int(m.group(1)) if m else None
    a, b = stunde(von), stunde(bis)
    if a is None or b is None:
        return None
    return set(range(a, b + 1)) if a <= b else set(range(a, 24)) | set(range(0, b + 1))


def tagesgang(meter, zeitraum, groesse="flow"):
    """Median je Tagesstunde im angegebenen Zeitraum."""
    df = zeitreihe(meter)
    if df is None:
        return None
    w = df.loc[zeitraum[0]:zeitraum[1]]
    return w.groupby(w.index.hour)[groesse].median() if len(w) else None


hz = beg[beg["heizzeit_geaendert"] & beg["meter"].notna()].copy()
print(f"Stationen mit dokumentierter Heizzeit-Aenderung: {len(hz)}\n")
print(f"{'Zaehler':>9}  {'Adresse':20s} {'alt':>12} {'neu':>12} | "
      f"{'weggefallen':>11} {'unveraendert':>12} {'dazugekommen':>12}")

hz_zeilen = []
for _, r in hz.iterrows():
    alt = fensterstunden(r["heizzeit_alt_von"], r["heizzeit_alt_bis"])
    neu = fensterstunden(r["heizzeit_neu_von"], r["heizzeit_neu_bis"])
    vor = tagesgang(int(r["meter"]), WINTER_VOR)
    nach = tagesgang(int(r["meter"]), WINTER_NACH)
    if not alt or not neu or vor is None or nach is None or vor.empty or nach.empty:
        continue
    aenderung = (nach - vor).dropna()

    def median_ueber(stunden_menge):
        werte = [aenderung[h] for h in stunden_menge if h in aenderung.index]
        return np.median(werte) if werte else np.nan

    weg, dazu, bleibt = median_ueber(alt - neu), median_ueber(neu - alt), median_ueber(alt & neu)
    hz_zeilen.append({"meter": int(r["meter"]), "adresse": r["key"],
                      "weggefallen": weg, "unveraendert": bleibt, "dazugekommen": dazu})
    a = f"{str(r['heizzeit_alt_von'])[:5]}-{str(r['heizzeit_alt_bis'])[:5]}"
    b = f"{str(r['heizzeit_neu_von'])[:5]}-{str(r['heizzeit_neu_bis'])[:5]}"
    print(f"{int(r['meter']):>9}  {r['key']:20s} {a:>12} {b:>12} | "
          f"{weg:11.0f} {bleibt:12.0f} {dazu:12.0f}")

hz_tab = pd.DataFrame(hz_zeilen)
mit_wegfall = hz_tab.dropna(subset=["weggefallen"])
bestaetigt = mit_wegfall["weggefallen"] < mit_wegfall["unveraendert"]
print(f"\nStationen mit weggefallenen Heizstunden: {len(mit_wegfall)}")
print(f"davon Durchfluss dort staerker gefallen als in unveraenderten Stunden: "
      f"{bestaetigt.sum()} von {len(mit_wegfall)}")

**Was macht der Code?**
- Zerlegt das Heizfenster in drei Stundenmengen: **weggefallen** (nur im alten Fenster),
  **unveraendert** (in beiden) und **dazugekommen** (nur im neuen)
- Vergleicht den Durchfluss-Tagesgang im Winter davor mit dem danach und misst die
  Aenderung getrennt fuer jede der drei Mengen
- Die Spalte „unveraendert“ ist die **eingebaute Kontrolle**: Sie faengt ab, was ohnehin
  passiert ist (Wetter, Verbrauch). Nur ein Unterschied zwischen „weggefallen“ und
  „unveraendert“ ist ein Effekt der Umstellung

**Warum nicht einfach die Lastspitze vergleichen**
Der erste Versuch nahm die Stunde mit der hoechsten Waermeabnahme (`idxmax`). Das war
untauglich: Bei den meisten Stationen bewegt sie sich gar nicht, bei zweien sprang sie um
9 bzw. 14 Stunden — der Tagesgang hat mehrere fast gleich hohe Stunden, und dann wackelt
das Maximum zufaellig. Ein Vergleich ueber „Stunden mit Durchfluss > 5 l/h“ war ebenso
untauglich, weil der Wert vorher wie nachher bei 95–100 % lag.

**Das Ergebnis**
Bei fuenf Stationen sind Heizstunden **weggefallen**. In vier davon ist der Durchfluss in
genau diesen Stunden staerker gesunken als in den unveraenderten — z. B. Zaehler 80912156
(06:00–22:00 → 07:00–21:00): −40 l/h in den weggefallenen Stunden bei +4 l/h in den
unveraenderten.

**Dazugekommene Stunden bestaetigen sich nicht.** Physikalisch plausibel: Ein Heizfenster
zu schliessen stoppt die Waermeabnahme zuverlaessig, es zu oeffnen erzwingt sie nicht — die
Raumregelung entscheidet weiter, ob geheizt wird.

Trotz kleiner Fallzahl ist das die belastbarste Einzelvalidierung im Notebook, weil die
Vorhersage **vor** dem Blick in die Daten feststand.

## 7 · Ergebnis speichern

In [ ]:
import json

ausgabe = ROOT / "data" / "processed" / "begehungs_wirkung.csv"
spalten = ["meter", "adresse", "datum", "kategorie", "von", "meter_gewechselt",
           "rt_vor", "rt_nach", "d_rt", "flow_vor", "flow_nach", "d_flow",
           "kontrolle", "did"]
wirkung[spalten].to_csv(ausgabe, index=False)

meta = {
    "erzeugt": pd.Timestamp.now().isoformat(timespec="seconds"),
    "n_begehungen_excel": int(len(beg)),
    "n_auswertbar": int(len(wirkung)),
    "n_kontrollgruppe": len(kontrollgruppe),
    "fenster_tage": int(FENSTER.days),
    "min_stunden": MIN_STUNDEN,
    "did_je_kategorie": {k: (None if pd.isna(v) else round(float(v), 2))
                         for k, v in tabelle["DiD"].items()},
}
ausgabe.with_suffix(".json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Wirkungstabelle -> {ausgabe}")
print(f"Lauf-Kontext    -> {ausgabe.with_suffix('.json')}")

**Was macht der Code?**
- Schreibt die Wirkungstabelle nach `data/processed/begehungs_wirkung.csv` — eine Zeile
  je auswertbarer Begehung, mit Roh-Aenderung, Kontrollwert und DiD
- `meter_gewechselt` wandert mit in den Export: Bei diesen Stationen kann der Vergleich
  zwei Geraete betreffen
- Daneben ein JSON mit dem Lauf-Kontext: Fenstergroesse, Mindeststundenzahl, Groesse der
  Kontrollgruppe, DiD je Kategorie

**Achtung beim Weitergeben:** Die CSV enthaelt Zaehlernummern und Adressen.
`data/processed/` ist gitignored und bleibt es — siehe Datenschutz-Abschnitt in
[`docs/README.md`](../docs/README.md).

## Zusammenfassung

| Befund | Ergebnis |
|---|---|
| Zuordnung Adresse → Zaehler | **66 von 66** ueber `Zuordnung.xlsx` (ueber `Nodes_Edges.ods`: 54) |
| Ruecklaufaenderung nach Kategorie | monoton ueber alle vier Stufen, „viele“ am staerksten |
| Statistische Signifikanz | **nicht erreicht** (Wilcoxon p ≈ 0,4) |
| Begehung im Zeitfenster nachweisbar | rund die Haelfte der Stationen mit Uhrzeit |
| Dokumentierte Heizzeit-Aenderung | in **4 von 5** Faellen mit weggefallenen Heizstunden bestaetigt |

**Warum dieser Ansatz noetig war**

Clustering ([`01_kmeans`](01_kmeans_stationscluster.ipynb)) fand keine Struktur
(Silhouette < 0,22), der ML-Klassifikator ([`docs/ml-klassifikator.md`](../docs/ml-klassifikator.md))
kein Signal. Gemeinsame Ursache ist das Label: `n_actions > 0` ist bei 76 % der Stationen wahr.

Der Beweis steckt in den Daten: Zaehler 68956351 hatte ein Ventil, das **drei Jahre offen
stand** (500–950 l/h durchgehend bei 0,1 kW) und am 09.09.2024 geschlossen wurde — ueber
Nacht von 499 auf 0 l/h. In der Begehungstabelle steht dazu **keine Massnahme**. Ein Label,
das die dramatischste Reparatur im Datensatz nicht kennt, kann kein Modell retten.

**Offene Punkte**

- **Zaehlerwechsel** bei fuenf Adressen — Vorher-Nachher kann dort zwei Geraete vergleichen
- **72167783**: Ruecklauf nach der Begehung von 18,6 auf 71,0 °C.
  Inbetriebnahme oder Fehler? Vor jeder Mittelwertbildung zu klaeren
- **Freitext kodieren**: 57 der 66 Begehungen haben eine Notiz mit dem tatsaechlichen Befund
  („Regler schliesst nicht ganz“, „Zirk Pumpe defekt geoelt“). Damit liessen sich die Faelle
  erklaeren, in denen Kategorie und Messergebnis auseinandergehen
- **Feature-Zeitfenster**: `hast_features.csv` mittelt ueber die gesamte Historie und
  beschreibt damit vergangene Zustaende — 68956351 gilt dort weiter als Problemfall